---
title: "Melbourne Property Price - Machine Learning"
author: "Hoang Son Lai"
date: "05/12/2026"
format: 
 html:
  toc: true
  css: styles.css
  embed-resources: true
  code-fold: true
---

## Introduction

This report builds, tunes and compares regression models to predict Melbourne property prices, then uses the best model to price the current For Sale supply.

Data snapshot: this report uses data updated on 2 May 2026.

The report consumes the cleaned dataset produced by `eda_report.ipynb`. All cleaning rules, outlier policies and encoding decisions were finalized in EDA and are loaded here from `eda_decisions.json` to keep the two reports in sync.

Three models are compared: Linear Regression with Ridge regularization (baseline), Random Forest, and XGBoost. The best model is then used to predict prices for all For Sale listings, with the current Year and Month injected at inference to obtain today-priced estimates. Two additional XGBoost quantile models (q=0.1, q=0.9) produce an 80% prediction interval for each listing.

Workflow: setup, problem framing, feature engineering, time-based 70/15/15 split, preprocessing pipeline, three model families, model comparison, SHAP interpretability, final retraining and For Sale inference, save artifacts.

## 1. Load cleaned data and EDA decisions

I load three things from the EDA output: the cleaned dataset (Sold + For Sale combined into one parquet), the JSON of decisions taken during EDA (split ratios, encoding rules, rare-type list, new-build categories), and a quick sanity check on group sizes.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import joblib

from sklearn.preprocessing  import StandardScaler
from sklearn.impute         import SimpleImputer
from sklearn.compose        import ColumnTransformer
from sklearn.pipeline       import Pipeline
from sklearn.linear_model   import LinearRegression, Ridge
from sklearn.ensemble       import RandomForestRegressor
from sklearn.metrics        import mean_squared_error, mean_absolute_error, r2_score

import xgboost as xgb
import shap

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["figure.figsize"] = (10, 5)

DATA_DIR  = Path("report_data")
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

# To convert to html, quarto render report/ml_report.ipynb

print("Setup complete.")

Setup complete.


In [2]:
# Load cleaned dataset and EDA decisions.
df = pd.read_parquet(DATA_DIR / "cleaned_data.parquet")

with open(DATA_DIR / "eda_decisions.json", "r") as f:
    decisions = json.load(f)

print(f"Loaded data: {df.shape}")
print(f"  Sold:     {(df['Status'] == 'Sold').sum():,}")
print(f"  For Sale: {(df['Status'] == 'For Sale').sum():,}")

print("\nDecision keys loaded:")
for k in decisions:
    print(f"  - {k}")

print(f"\nSnapshot date: {decisions['data_snapshot_date']}")
print(f"Split ratios:  {decisions['split_ratios']}")
print(f"Target transform: {decisions['target_transform']}")
print(f"Inference Year/Month: {decisions['inference_year']}/{decisions['inference_month']}")

Loaded data: (163239, 32)
  Sold:     124,820
  For Sale: 38,419

Decision keys loaded:
  - data_snapshot_date
  - land_property_types
  - apt_like_types
  - rural_large_types
  - residential_dense_types
  - price_floor
  - price_ceiling
  - landsize_caps_by_type
  - metro_envelope
  - target_transform
  - split_strategy
  - split_ratios
  - inference_year
  - inference_month
  - rare_property_types
  - new_build_types
  - rationale_year_month

Snapshot date: 2026-05-02
Split ratios:  {'train': 0.7, 'val': 0.15, 'test': 0.15}
Target transform: log1p
Inference Year/Month: 2026/5


## 2. Problem framing

**Goal:** predict `Numeric_Price` for For Sale listings.

**Training set:** Sold rows that have a valid (non-NaN) `Numeric_Price`. Rows with "Price Withheld" or with any other missing target are excluded from training.

**Inference set:** all For Sale rows.

**Target:** `log1p(Numeric_Price)`. The log transform was justified in EDA Section 4.1 by the heavy right-skew of raw price.

**Time-aware features:** each Sold row uses its actual Year and Month at training time, letting the model absorb the 4-12% YoY inflation observed in EDA Section 6. For Sale rows have no transaction date by structural design, so at inference I inject Year = 2026, Month = 5 (the snapshot date) so predictions reflect today's market level rather than the historical average.

**Evaluation:** I report RMSE, MAE, MAPE, and R² on both log scale and original AUD scale. RMSE on AUD is the primary metric because absolute pricing error is what matters for downstream decisions.

In [3]:
# Split the combined frame into training-eligible Sold and inference-target For Sale.
df_sold    = df[df["Status"] == "Sold"].copy()
df_forsale = df[df["Status"] == "For Sale"].copy()

# Training set requires a non-NaN Numeric_Price.
df_train_pool = df_sold.dropna(subset=["Numeric_Price"]).copy()

print(f"Sold total:             {len(df_sold):,}")
print(f"Sold with valid price:  {len(df_train_pool):,}  (training pool)")
print(f"For Sale (inference):   {len(df_forsale):,}")

print(f"\nTraining pool date range: "
      f"{df_train_pool['Date_parsed'].min().date()} to "
      f"{df_train_pool['Date_parsed'].max().date()}")
print(f"\nTarget summary (raw AUD):")
print(df_train_pool["Numeric_Price"].describe(percentiles=[.25, .5, .75, .95]).round(0))

Sold total:             124,820
Sold with valid price:  114,594  (training pool)
For Sale (inference):   38,419

Training pool date range: 2005-12-01 to 2026-05-02

Target summary (raw AUD):
count      114594.0
mean       958510.0
std        701764.0
min         50000.0
25%        610000.0
50%        772295.0
75%       1081545.0
95%       2100000.0
max      19500000.0
Name: Numeric_Price, dtype: float64
